In [3]:

import sqlite3
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_validate
from sklearn.linear_model import LinearRegression
from sklearn.metrics import make_scorer, mean_squared_error

# Load data
conn = sqlite3.connect("/Users/Maazuddin/Downloads/inventory.db")

df = pd.read_sql_query(
    """
    SELECT Dollars, Quantity, Freight
    FROM vendor_invoice
    """,
    conn
)

conn.close()

# Clean data
df = df.dropna(
    subset=["Dollars", "Quantity", "Freight"]
)

X = df[["Dollars", "Quantity"]]
y = df["Freight"]

# Define 5-fold cross-validation
kfold = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

model = LinearRegression()

# Evaluate MAE, RMSE, and R²
scoring = {
    "mae": "neg_mean_absolute_error",
    "r2": "r2",
    "neg_mse": "neg_mean_squared_error"
}

results = cross_validate(
    model,
    X,
    y,
    cv=kfold,
    scoring=scoring,
    return_train_score=False
)

mae_scores = -results["test_mae"]
rmse_scores = np.sqrt(-results["test_neg_mse"])
r2_scores = results["test_r2"]

print("\n5-Fold Cross-Validation Results")
print("--------------------------------")

for i in range(5):
    print(
        f"Fold {i + 1}: "
        f"MAE = {mae_scores[i]:.4f}, "
        f"RMSE = {rmse_scores[i]:.4f}, "
        f"R² = {r2_scores[i]:.4f}"
    )

print("\nAverage Scores")
print(f"Mean MAE:  {mae_scores.mean():.4f}")
print(f"Mean RMSE: {rmse_scores.mean():.4f}")
print(f"Mean R²:   {r2_scores.mean():.4f}")

print("\nScore Variability (Standard Deviation)")
print(f"MAE Std:  {mae_scores.std():.4f}")
print(f"RMSE Std: {rmse_scores.std():.4f}")
print(f"R² Std:   {r2_scores.std():.4f}")



5-Fold Cross-Validation Results
--------------------------------
Fold 1: MAE = 24.4592, RMSE = 124.4288, R² = 0.9700
Fold 2: MAE = 26.5629, RMSE = 127.3900, R² = 0.9691
Fold 3: MAE = 25.4286, RMSE = 94.2010, R² = 0.9810
Fold 4: MAE = 25.8071, RMSE = 162.8754, R² = 0.9468
Fold 5: MAE = 23.3124, RMSE = 88.5294, R² = 0.9855

Average Scores
Mean MAE:  25.1140
Mean RMSE: 119.4849
Mean R²:   0.9705

Score Variability (Standard Deviation)
MAE Std:  1.1272
RMSE Std: 26.7102
R² Std:   0.0134
